In [1]:
from pyspark.sql import SparkSession
from xgboost.spark import SparkXGBClassifier
from pyspark.ml.linalg import Vectors

In [2]:
spark = SparkSession\
    .builder\
    .appName("SparkXGBoostClassifier Example")\
    .config("spark.hadoop.fs.s3a.s3guard.ddb.region", "us-east-2")\
    .config("spark.kerberos.access.hadoopFileSystems","s3a://cf-buk-7b7cacc6/data")\
    .config("spark.dynamicAllocation.enabled", "false")\
    .config("spark.executor.cores", "4")\
    .config("spark.executor.memory", "4g")\
    .config("spark.executor.instances", "4")\
    .config("spark.driver.core","4")\
    .config("spark.driver.memory","4g")\
    .getOrCreate()

Setting spark.hadoop.yarn.resourcemanager.principal to pauldefusco


In [3]:
import os
print("https://spark-"+os.environ["CDSW_ENGINE_ID"]+"."+os.environ["CDSW_DOMAIN"])

https://spark-nbhgtnf7fs6rm84a.ml-44d87eed-e8f.cf-cdp-e.a465-9q4k.cloudera.site


In [4]:
df_train = spark.createDataFrame([
    (Vectors.dense(1.0, 2.0, 3.0), 0, False, 1.0),
    (Vectors.sparse(3, {1: 1.0, 2: 5.5}), 1, False, 2.0),
    (Vectors.dense(4.0, 5.0, 6.0), 0, True, 1.0),
    (Vectors.sparse(3, {1: 6.0, 2: 7.5}), 1, True, 2.0),
], ["features", "label", "isVal", "weight"])

In [5]:
df_train.collect()

[Row(features=DenseVector([1.0, 2.0, 3.0]), label=0, isVal=False, weight=1.0),
 Row(features=SparseVector(3, {1: 1.0, 2: 5.5}), label=1, isVal=False, weight=2.0),
 Row(features=DenseVector([4.0, 5.0, 6.0]), label=0, isVal=True, weight=1.0),
 Row(features=SparseVector(3, {1: 6.0, 2: 7.5}), label=1, isVal=True, weight=2.0)]

In [6]:
df_test = spark.createDataFrame([
    (Vectors.dense(1.0, 2.0, 3.0), ),
], ["features"])

In [7]:
xgb_classifier = SparkXGBClassifier(max_depth=5, missing=0.0,
    validation_indicator_col='isVal', weight_col='weight',
    early_stopping_rounds=1, eval_metric='logloss', num_workers=2)

In [8]:
xgb_clf_model = xgb_classifier.fit(df_train)

/home/cdsw/.local/lib/python3.10/site-packages/xgboost/sklearn.py:782: UserWarning: Loading a native XGBoost model with Scikit-Learn interface.
  warnings.warn("Loading a native XGBoost model with Scikit-Learn interface.")


In [9]:
xgb_clf_model.transform(df_test).show()

[Stage 16:==============================================>          (9 + 2) / 11]

+-------------+-------------+----------+-----------+
|     features|rawPrediction|prediction|probability|
+-------------+-------------+----------+-----------+
|[1.0,2.0,3.0]|   [-0.0,0.0]|       0.0|  [0.5,0.5]|
+-------------+-------------+----------+-----------+



In [10]:
!pip install onnx onnxmltools skl2onnx

In [14]:
import xgboost as xgb
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType

# Save booster temporarily
booster = xgb_clf_model.get_booster()
booster.save_model("/home/cdsw/xgb.json")

# Re-load into standard XGBoost classifier
native_model = xgb.XGBClassifier()
native_model.load_model("/home/cdsw/xgb.json")

In [15]:
import numpy as np

pdf = df_train.select("features").toPandas()

X = np.vstack(pdf["features"].apply(lambda v: v.toArray()).values)

print(X)
print(X.shape)

[[1.  2.  3. ]
 [0.  1.  5.5]
 [4.  5.  6. ]
 [0.  6.  7.5]]
(4, 3)


In [16]:
preds = native_model.predict(X)
proba = native_model.predict_proba(X)

In [17]:
preds

array([0, 0, 0, 0])

In [18]:
proba

array([[0.5, 0.5],
       [0.5, 0.5],
       [0.5, 0.5],
       [0.5, 0.5]], dtype=float32)